In [ ]:
from datetime import date, timedelta
from pyspark.sql import Row, functions as F
import json

spark.conf.set("spark.sql.parquet.vorder.default", "true")
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.optimizeWrite.binSize", "1g")

GOLD_ACTIVITIES = "FactWorkout"
GOLD_WEEKLY = "FactTrainingWeek"
GOLD_WORKOUT_PLAN = "FactWorkoutPlan"
LEGACY_GOLD_TABLES = ["workout_plan"]

goal = "balanced fitness"
available_days = 5
plan_days = 7

activities = spark.read.format("delta").load(f"Tables/{GOLD_ACTIVITIES}").select(
    F.col("activity_date"),
    F.col("type").alias("activity_type"),
    F.col("duration_minutes"),
    F.col("distance_km"),
    F.col("average_heartrate").alias("avg_heart_rate_bpm"),
    F.col("max_heartrate").alias("max_heart_rate_bpm"),
    F.col("average_watts").alias("avg_power_watts"),
    F.col("suffer_score").alias("training_load"),
)
weekly = spark.read.format("delta").load(f"Tables/{GOLD_WEEKLY}")

print("Generating gold workout plan from standardized silver activities and gold weekly metrics")

In [ ]:
summary = activities.agg(
    F.max("activity_date").alias("last_activity_date"),
    F.count("*").alias("total_activities"),
    F.avg("duration_minutes").alias("avg_duration_min"),
    F.expr("percentile_approx(duration_minutes, 0.75)").alias("p75_duration_min"),
    F.max("duration_minutes").alias("max_duration_min"),
    F.avg("avg_heart_rate_bpm").alias("avg_hr"),
    F.max("max_heart_rate_bpm").alias("observed_max_hr"),
    F.avg("avg_power_watts").alias("avg_watts"),
).collect()[0]

last_date = summary["last_activity_date"] or date.today()
recent_28 = activities.filter(F.col("activity_date") >= F.date_sub(F.lit(last_date), 28))
recent_7 = activities.filter(F.col("activity_date") >= F.date_sub(F.lit(last_date), 7))
previous_7 = activities.filter((F.col("activity_date") < F.date_sub(F.lit(last_date), 7)) & (F.col("activity_date") >= F.date_sub(F.lit(last_date), 14)))

recent = recent_28.agg(
    F.count("*").alias("sessions_28d"),
    F.sum("duration_minutes").alias("minutes_28d"),
    F.sum("distance_km").alias("distance_28d"),
    F.sum(F.coalesce("training_load", F.lit(0.0))).alias("load_28d"),
).collect()[0]

load_7 = recent_7.agg(F.sum(F.coalesce("training_load", F.lit(0.0))).alias("load")).collect()[0]["load"] or 0.0
load_previous_7 = previous_7.agg(F.sum(F.coalesce("training_load", F.lit(0.0))).alias("load")).collect()[0]["load"] or 0.0
ramp_pct = None if load_previous_7 == 0 else round((load_7 - load_previous_7) / load_previous_7 * 100.0, 1)

sport_rows = (
    recent_28.groupBy("activity_type")
    .agg(F.count("*").alias("sessions"), F.sum("duration_minutes").alias("minutes"), F.sum("distance_km").alias("distance_km"))
    .orderBy(F.desc("minutes"))
    .collect()
)
main_sport = sport_rows[0]["activity_type"] if sport_rows else "Ride"
if main_sport == "VirtualRide":
    main_sport = "Ride"

latest_week = weekly.orderBy(F.desc("week_start")).limit(1).collect()
readiness_flag = latest_week[0]["readiness_flag"] if latest_week else "Steady"

avg_duration = float(summary["avg_duration_min"] or 45)
p75_duration = float(summary["p75_duration_min"] or avg_duration or 60)
weekly_minutes = float(recent["minutes_28d"] or (available_days * avg_duration)) / 4.0

if readiness_flag == "Ramp too fast" or (ramp_pct is not None and ramp_pct > 25):
    target_minutes = weekly_minutes * 0.75
    week_focus = "Recovery week: recent load ramp is high. Keep intensity controlled."
elif goal == "recovery":
    target_minutes = weekly_minutes * 0.65
    week_focus = "Recovery week: maintain movement and reduce fatigue."
else:
    target_minutes = weekly_minutes * 1.05
    week_focus = "Build week: progress gradually with mostly easy aerobic volume."

easy_min = max(30, round(target_minutes * 0.18 / 5) * 5)
endurance_min = max(45, round(target_minutes * 0.25 / 5) * 5)
long_min = max(60, round(min(p75_duration * 1.25, target_minutes * 0.35) / 5) * 5)
interval_min = max(40, round(target_minutes * 0.18 / 5) * 5)
strength_min = 40

observed_max_hr = summary["observed_max_hr"]
if observed_max_hr:
    z2_hr = f"HR Z2 approx {round(observed_max_hr * 0.60)}-{round(observed_max_hr * 0.72)} bpm"
    tempo_hr = f"tempo approx {round(observed_max_hr * 0.78)}-{round(observed_max_hr * 0.86)} bpm"
else:
    z2_hr = "easy conversational effort (RPE 3-4/10)"
    tempo_hr = "comfortably hard tempo (RPE 7/10)"

avg_watts = summary["avg_watts"]
if avg_watts:
    endurance_power = f"power target about {round(avg_watts * 0.80)}-{round(avg_watts * 0.95)} W"
    interval_power = f"hard reps around {round(avg_watts * 1.15)}-{round(avg_watts * 1.30)} W"
else:
    endurance_power = "steady endurance effort"
    interval_power = "hard but repeatable reps"

start = date.today() + timedelta(days=1)
workout_templates = [
    (0, "Recovery mobility", "Recovery", 25, "Yoga / mobility + easy walk", "Keep it very easy. Focus hips, calves, hamstrings, thoracic spine."),
    (1, f"Easy {main_sport} endurance", "Endurance", easy_min, f"{z2_hr}; {endurance_power}", "Relaxed aerobic session. You should finish fresher than you started."),
    (2, "Strength + core", "Strength", strength_min, "Full-body strength", "3 rounds: squats/lunges, hinge, push, pull, plank. Stop 2 reps before failure."),
    (3, f"{main_sport} intervals", "Quality", interval_min, f"Warm up 10 min; 5 x 3 min hard / 3 min easy; {interval_power}; cool down", "Use this only if legs feel good. Otherwise swap for easy endurance."),
    (4, "Rest day", "Rest", 0, "No structured training", "Walk optional. Prioritize sleep and nutrition."),
    (5, f"Long {main_sport} endurance", "Long endurance", long_min, f"{z2_hr}; fuel from minute 45", "Practice fueling: 45-75g carbs/hour for sessions over 90 min."),
    (6, "Tempo aerobic progression", "Tempo", endurance_min, f"15 min easy + 2 x 10 min {tempo_hr} + easy cool down", "Controlled tempo, not a race. Keep form smooth."),
]

if readiness_flag == "Ramp too fast" or (ramp_pct is not None and ramp_pct > 25):
    workout_templates[3] = (3, f"Easy {main_sport} spin/run", "Endurance", easy_min, z2_hr, "Intervals replaced because your recent load ramp is high.")
    workout_templates[6] = (6, "Easy aerobic + mobility", "Recovery", 35, "Easy effort only", "Keep this restorative.")

rows = []
for offset, workout_name, category, duration, target, notes in workout_templates[:plan_days]:
    plan_date = start + timedelta(days=offset)
    rows.append(Row(
        plan_date=plan_date,
        day_name=plan_date.strftime("%a"),
        workout_name=workout_name,
        category=category,
        duration_minutes=int(duration),
        target=target,
        notes=notes,
        based_on_main_sport=main_sport,
        week_focus=week_focus,
        readiness_flag=readiness_flag,
        recent_7d_load=float(load_7),
        previous_7d_load=float(load_previous_7),
        ramp_pct=float(ramp_pct) if ramp_pct is not None else None,
        generated_at=date.today(),
    ))

plan_df = spark.createDataFrame(rows).withColumn("date_key", F.col("plan_date"))
plan_df.write.mode("overwrite").format("delta").option("overwriteSchema", "true").save(f"Tables/{GOLD_WORKOUT_PLAN}")
spark.sql(f"OPTIMIZE delta.`Tables/{GOLD_WORKOUT_PLAN}`")
for legacy in LEGACY_GOLD_TABLES:
    if spark.catalog.tableExists(legacy):
        spark.sql(f"DROP TABLE {legacy}")
        print(f"Dropped legacy Gold table {legacy}")

result = {
    "status": "success",
    "table": GOLD_WORKOUT_PLAN,
    "rows": plan_df.count(),
    "main_sport": main_sport,
    "readiness_flag": readiness_flag,
    "week_focus": week_focus,
}

print(json.dumps(result, indent=2))
plan_df.show(truncate=False)

notebookutils.notebook.exit(json.dumps(result))